<a href="https://colab.research.google.com/github/bajon1/Deep-learning-lab/blob/main/Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna statsmodels -q

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, KFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import optuna
from statsmodels.stats.contingency_tables import cochrans_q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 5.5 MB/s eta 0:00:00


# Lab 3: MLP – trening configuration

# Task 1: MLP implementation for MNIST

#### Goal: Build a multi-layer network to classify flattened MNIST images.

```python
# Load flattened MNIST images (vectors of dimension 784) from scikit-learn
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist["data"], mnist["target"]
X = X.astype(np.float32) / 255.0 # scale to [0, 1]
y = y.astype(np.int64)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=10000, random_state=42, stratify=y
)
```

1. Define an MLP with architecture: input=784 → hidden=1568 → hidden=1568 → output=10.
2. Use SGD optimiser, CrossEntropyLoss, and a learning rate of 0.01.
3. Split the trainval set into five folds using 5-fold cross-validation.
4. Train in mini-batches of 1024 images; each epoch consumes all training images. Set the model to model.train() mode during training.
5. After each training epoch, switch to model.eval() mode, run inference on the validation fold, and compute the mean validation loss. Whenever the validation loss reaches a new minimum, save a checkpoint containing: model state dict, optimiser state dict, epoch number, best validation loss, and the full list of training/validation losses across all completed epochs.
6. Plot validation and training loss as a function of epoch.
7. Train a separate model for each fold — the result is five independent models used in Task 2.

# Task 2: Inference and prediction uncertainty

#### Goal: Evaluate prediction uncertainty using ensemble methods.

1. Majority voting — run all five models, obtain the argmax class from each, and return the class chosen by the majority.
2. Mean response — run all five models, average the softmax probabilities across models, and return the argmax of the averaged distribution.
3. Use Shannon entropy to estimate prediction uncertainty for both ensemble variants. How does entropy differ between confident and uncertain predictions?
4. Report test-set accuracy for both variants.
5. Use Cochran's Q test to determine whether the accuracy difference between the two methods is statistically significant.

# Task 3: Learning rate selection

#### Goal: Find an optimal learning rate using automated hyperparameter search.

1. Implement an objective_lr function for Optuna that trains the network for a small number of epochs and returns the mean 5-fold cross-validation loss on the validation set.
2. Run the Optuna study to find the optimal learning rate.
3. Retrain the full model using the optimal learning rate found, following the same procedure as Task 1.
4. Compare loss curves between the fixed LR from Task 1 and the Optuna-optimised LR using a side-by-side plot.

# Task 4: Optimiser comparison

#### Goal: Investigate how the choice of optimiser affects training performance.

1. Train the model with SGD using default PyTorch parameters.
2. Train the model with Adam using default PyTorch parameters.
3. Use Optuna to optimise SGD hyperparameters: momentum, nesterov, and learning_rate.
4. Use Optuna to optimise Adam hyperparameters: betas and learning_rate.
5. Compare all four configurations — plot training and validation loss curves for each.

# Task 5: Dropout

#### Goal: Explore Dropout as a regulariser and as a tool for uncertainty estimation.

1. Add nn.Dropout layers to the MLP from Task 1.
2. Train using the same procedure as Task 1.
3. Evaluate whether adding dropout improves test-set accuracy. Use the Mean Response ensemble strategy with the model in model.eval() mode.
4. Monte Carlo Dropout — keep the model in model.train() mode during inference and query the model multiple times for the same fixed input. Observe how the output distribution varies. Estimate prediction uncertainty for: (a) an input that IS a handwritten digit, and (b) an input that is NOT a handwritten digit.